# 01 | Filing Ingestion (FinanceBench corpus)

sec.gov is not reachable from the build environment, so filings come from the FinanceBench open-source corpus: passages from real 10-K, 10-Q, 8-K filings and earnings releases. One row per unique filing. Coverage is 100% by construction because the corpus is pre-indexed; this measures corpus content, not live EDGAR retrieval.

In [1]:
import sys, os
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

from src.paths import RAW, PROCESSED, RESULTS
from src import setup_data

RESULTS.mkdir(exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
if not setup_data.data_present():
    setup_data.main()

/usr/local/lib/python3.12/dist-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [2]:
import pandas as pd
from src.ingestion import ingest_financebench, summarize

FB = RAW / "financebench" / "financebench_open_source.jsonl"
assert FB.exists(), f"Missing {FB}; run python src/setup_data.py"

df_filings = ingest_financebench(FB)
m = summarize(df_filings)

print(f"Filings in corpus: {m['n_total']} ({m['unique_companies']} companies, {m['total_questions']} QA pairs)")
print(f"Parsed: {m['n_parsed']}/{m['n_total']} ({m['coverage_pct']}%, pre-indexed corpus)")
print(f"Filing types: {m['filing_types']}")
print(f"Avg evidence length: {m['avg_evidence_length']:,.0f} chars")
print(f"Avg parse time: {m['avg_parse_time_sec'] * 1000:.3f} ms per filing (local JSON, no download)")
print(f"Revenue mentioned: {df_filings['revenue_mentioned'].mean():.0%} | EPS mentioned: {df_filings['eps_mentioned'].mean():.0%}")
print(m["note"])

Filings in corpus: 84 (32 companies, 150 QA pairs)
Parsed: 84/84 (100.0%, pre-indexed corpus)
Filing types: {'10-K': 64, '10-Q': 8, '8-K': 6, 'EARNINGS': 6}
Avg evidence length: 2,914 chars
Avg parse time: 0.009 ms per filing (local JSON, no download)
Revenue mentioned: 52% | EPS mentioned: 12%
Source: FinanceBench pre-indexed SEC corpus (sec.gov blocked in sandbox); all 150 QA pairs span real filings


In [3]:
df_filings.to_csv(RESULTS / "filings_clean.csv", index=False)
df_filings.to_csv(PROCESSED / "filings_clean.csv", index=False)  # dashboard reads this path
summary = {k: v for k, v in m.items() if k != "filing_types"}
summary.update({f"n_{k.replace('-', '').lower()}": v for k, v in m["filing_types"].items()})
pd.DataFrame([summary]).to_csv(RESULTS / "ingestion_summary.csv", index=False)
print("Saved results/filings_clean.csv and results/ingestion_summary.csv")
df_filings.head(10)

Saved results/filings_clean.csv and results/ingestion_summary.csv


,ticker,doc_name,filing_type,period_end,revenue_mentioned,eps_mentioned,parse_success,parse_time_sec,evidence_length,question_count
0,3M,3M_2018_10K,10-K,2018,False,False,True,0.000031,5114,2
1,3M,3M_2022_10K,10-K,2022,False,False,True,0.000017,3198,3
2,3M,3M_2023Q2_10Q,10-Q,2023Q2,False,False,True,0.000010,1590,3
3,AES Corporation,AES_2022_10K,10-K,2022,True,True,True,0.000014,7905,3
4,AMD,AMD_2015_10K,10-K,2015,True,False,True,0.000007,3225,1
5,AMD,AMD_2022_10K,10-K,2022,True,False,True,0.000045,5910,7
6,Activision Blizzard,ACTIVISIONBLIZZARD_2019_10K,10-K,2019,True,False,True,0.000015,5998,2
7,Adobe,ADOBE_2015_10K,10-K,2015,True,False,True,0.000030,12125,1
8,Adobe,ADOBE_2016_10K,10-K,2016,True,False,True,0.000006,1691,1
9,Adobe,ADOBE_2017_10K,10-K,2017,True,False,True,0.000011,4999,1
